In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


# =====================================================
# MLP policy 3 → 16 → 1
# =====================================================

class MLP(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3,16)
        self.fc2 = nn.Linear(16,1)

    def forward(self,x):
        h = F.relu(self.fc1(x))
        return self.fc2(h)


# =====================================================
# Value network
# =====================================================

class ValueNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1,32)
        self.fc2 = nn.Linear(32,1)

    def forward(self,x):
        h = F.relu(self.fc1(x))
        return self.fc2(h)


# =====================================================
# Initial state
# =====================================================

def init_state(N):

    X = torch.zeros(N,N)

    for i in range(N):
        X[i,i] = 1

    return X


# =====================================================
# Free capacities
# =====================================================

def compute_free_capacities(X,weights,W):

    load = torch.matmul(weights,X)

    return W - load


# =====================================================
# Energy
# =====================================================

def energy(X):

    used_bins = torch.sum(X,dim=0) > 0

    return torch.sum(used_bins)


# =====================================================
# Select item
# =====================================================

def select_item(policy,X,weights,free_caps,T):

    N = X.shape[0]

    bin_indices = torch.argmax(X,dim=1)

    c_bi = free_caps[bin_indices]

    T_vec = torch.full((N,),T)

    features = torch.stack([weights,c_bi,T_vec],dim=1)

    logits = policy(features).squeeze(-1)

    probs = torch.softmax(logits,dim=0)

    dist = torch.distributions.Categorical(probs)

    i = dist.sample()

    return i, dist.log_prob(i)


# =====================================================
# Select bin
# =====================================================

def select_bin(policy,i,X,weights,free_caps,T):

    N = X.shape[0]

    w_i = weights[i]

    weight_vec = torch.full((N,),w_i)

    T_vec = torch.full((N,),T)

    features = torch.stack([weight_vec,free_caps,T_vec],dim=1)

    logits = policy(features).squeeze(-1)

    mask = free_caps >= w_i

    logits[~mask] = -1e9

    probs = torch.softmax(logits,dim=0)

    dist = torch.distributions.Categorical(probs)

    j = dist.sample()

    return j, dist.log_prob(j)


# =====================================================
# Apply move
# =====================================================

def apply_move(X,i,j):

    X_new = X.clone()

    old_bin = torch.argmax(X_new[i])

    X_new[i,old_bin] = 0

    X_new[i,j] = 1

    return X_new


# =====================================================
# Metropolis
# =====================================================

def metropolis_step(X,X_new,T):

    E_old = energy(X)

    E_new = energy(X_new)

    delta = E_new - E_old

    if delta <= 0:
        return X_new

    prob = torch.exp(-delta/T)

    if torch.rand(1) < prob:
        return X_new

    return X


# =====================================================
# Returns
# =====================================================

def compute_returns(rewards,gamma=0.99):

    returns = []

    G = 0

    for r in reversed(rewards):

        G = r + gamma*G

        returns.insert(0,G)

    return torch.tensor(returns,dtype=torch.float32)


# =====================================================
# Rollout
# =====================================================

def rollout(X,weights,W,steps,item_policy,bin_policy):

    log_probs = []
    rewards = []
    states_energy = []
    actions = []
    states_X = []

    for t in range(steps):

        states_X.append(X.clone())

        T = 1.0 * (1 - t/steps)

        free_caps = compute_free_capacities(X,weights,W)

        i,logp_i = select_item(item_policy,X,weights,free_caps,T)

        j,logp_j = select_bin(bin_policy,i,X,weights,free_caps,T)

        X_new = apply_move(X,i,j)

        E_old = energy(X)

        X = metropolis_step(X,X_new,T)

        E_new = energy(X)

        reward = (E_old - E_new).item()

        log_probs.append(logp_i + logp_j)

        rewards.append(reward)

        states_energy.append(torch.tensor([E_new],dtype=torch.float32))

        actions.append((i,j))

    return X, log_probs, rewards, states_energy, actions, states_X


# =====================================================
# PPO update
# =====================================================

def ppo_update(item_policy,
               bin_policy,
               value_net,
               optimizer,
               log_probs_old,
               rewards,
               states_energy,
               actions,
               states_X,
               weights,
               W):

    returns = compute_returns(rewards)

    values = torch.cat([value_net(s) for s in states_energy]).squeeze()

    advantages = returns - values.detach()

    log_probs_old = torch.stack(log_probs_old)

    new_log_probs = []

    for t in range(len(actions)):

        X = states_X[t]

        i,j = actions[t]

        free_caps = compute_free_capacities(X,weights,W)

        T = 1.0

        _, logp_i = select_item(item_policy,X,weights,free_caps,T)

        _, logp_j = select_bin(bin_policy,i,X,weights,free_caps,T)

        new_log_probs.append(logp_i + logp_j)

    new_log_probs = torch.stack(new_log_probs)

    ratio = torch.exp(new_log_probs - log_probs_old)

    surr1 = ratio * advantages

    surr2 = torch.clamp(ratio,0.8,1.2) * advantages

    policy_loss = -torch.min(surr1,surr2).mean()

    value_loss = (returns - values).pow(2).mean()

    loss = policy_loss + 0.5*value_loss

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    return loss.item()


# =====================================================
# Training
# =====================================================

N = 10

weights = torch.tensor([2.0,3.0,1.5,4.0,2.5,3.5,2.2,1.7,2.8,3.1])

W = 6.0

steps = 100

item_policy = MLP()

bin_policy = MLP()

value_net = ValueNet()

optimizer = optim.Adam(
    list(item_policy.parameters()) +
    list(bin_policy.parameters()) +
    list(value_net.parameters()),
    lr=1e-3
)

X = init_state(N)

best_bins = float("inf")
best_X = None
best_epoch = 0


for epoch in range(200):

    X, log_probs, rewards, states_energy, actions, states_X = rollout(
        X,
        weights,
        W,
        steps,
        item_policy,
        bin_policy
    )

    loss = ppo_update(
        item_policy,
        bin_policy,
        value_net,
        optimizer,
        log_probs,
        rewards,
        states_energy,
        actions,
        states_X,
        weights,
        W
    )

    current_bins = energy(X).item()

    if current_bins < best_bins:

        best_bins = current_bins
        best_X = X.clone()
        best_epoch = epoch

    if epoch % 20 == 0:

        print(
            "epoch", epoch,
            "| bins", current_bins,
            "| best", best_bins,
            "| loss", loss
        )


print("\n===== BEST SOLUTION =====")

print("best bins:", best_bins)

print("found at epoch:", best_epoch)

print("best X:\n", best_X)

epoch 0 | bins 6 | best 6 | loss 2.54797625541687
epoch 20 | bins 6 | best 5 | loss -0.3679547905921936
epoch 40 | bins 6 | best 5 | loss 0.06523731350898743
epoch 60 | bins 5 | best 5 | loss -0.05903923511505127
epoch 80 | bins 5 | best 5 | loss 0.05765897035598755
epoch 100 | bins 6 | best 5 | loss 0.6030325293540955
epoch 120 | bins 6 | best 5 | loss 0.42694348096847534
epoch 140 | bins 5 | best 5 | loss 0.3379489779472351
epoch 160 | bins 5 | best 5 | loss -0.01119697093963623
epoch 180 | bins 5 | best 5 | loss 0.15214185416698456

===== BEST SOLUTION =====
best bins: 5
found at epoch: 1
best X:
 tensor([[0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0